# Data Lake Übung — Berliner Verkehrsbetriebe

Du bist neu eingestellte:r Data Engineer bei einem fiktiven Berliner Mobilitätsanbieter. Aus rohen Buchungsdaten sollst du eine analytisch nutzbare Datenbasis aufbauen — vom chaotischen Raw Layer bis zum partitionierten Silver Layer.

**Dauer:** ca. 30 Minuten, **5 Aufgaben + 1 optionale**, jede baut auf der vorherigen auf.

**Was du lernst:** Glob-Pattern, `DESCRIBE`, Schema-Drift mit `union_by_name`, `COPY ... TO ... (FORMAT parquet)`, Performance-Vergleich CSV vs. Parquet, Multi-Format-Joins, Partitionierung.

**Spielregel:** Du füllst **nur** die SQL-Strings in den `con.sql(""" ... """)`-Blöcken aus. Aller Python-Drumherum ist vorgegeben.

---

## Setup

Lade die bereitgestellte **`raw.zip`** hoch — die Zelle entpackt sie in dein Arbeitsverzeichnis.

In [ ]:
!pip install duckdb pyarrow -q

In [ ]:
import os, zipfile

# In Colab: Upload-Dialog. Lokal: einfach raw.zip neben das Notebook legen.
if not os.path.isdir("raw"):
    if not os.path.exists("raw.zip"):
        try:
            from google.colab import files  # type: ignore
            print("Bitte raw.zip hochladen:")
            files.upload()
        except ImportError:
            raise SystemExit(
                "Bitte raw.zip neben das Notebook legen oder in Colab hochladen."
            )
    with zipfile.ZipFile("raw.zip") as z:
        z.extractall(".")

os.makedirs("silver", exist_ok=True)

print("Lake-Inhalt:")
for root, _, fs in os.walk("raw"):
    rel = os.path.relpath(root, ".")
    print(f"  {rel}/")
    for f in sorted(fs):
        size_kb = os.path.getsize(os.path.join(root, f)) / 1024
        print(f"    {f}  ({size_kb:,.1f} KB)")

In [ ]:
import duckdb
con = duckdb.connect()
print("DuckDB-Version:", duckdb.__version__)

---

## Aufgabe 1 — Den Lake erkunden (Glob + `DESCRIBE`)

**Story:** Erster Tag im Job. Im Lake liegen 6 Wochendateien mit Buchungen. Du hast gehört, dass die letzten Wochen "irgendwie anders sind", und konzentrierst dich daher erstmal auf die ersten 4 Wochen.

**Aufgaben:**
1. Lies die ersten 4 Wochendateien mit einem **einzigen** SQL-Statement und zähle die Buchungen.
2. Zeige, wie viele Zeilen jede einzelne Wochendatei beisteuert (Pseudospalte `filename`).
3. Lass dir das Schema einer Wochendatei mit `DESCRIBE` ausgeben.

**Hints:**
- DuckDB versteht Glob-Pattern direkt im Dateipfad — auch Zeichenklassen wie `[1-4]` funktionieren. Damit erübrigt sich jedes `UNION`.
- `read_csv()` hat eine Option, die den Dateinamen als zusätzliche Spalte einblendet — sehr nützlich, um zu sehen, woher eine Zeile kommt.
- `DESCRIBE` lässt sich vor jedes `SELECT` oder `FROM` setzen.

**1.1** — Gesamtzahl Buchungen aus den ersten 4 Wochen.

In [ ]:
con.sql("""
    SELECT COUNT(*) AS gesamt
    FROM 'raw/buchungen/woche_0[1-4].csv'
""").show()

**1.2** — Zeilen pro Datei mit der `filename`-Pseudospalte.

In [ ]:
con.sql("""
    SELECT filename, COUNT(*) AS zeilen
    FROM read_csv('raw/buchungen/woche_0[1-4].csv', filename=true)
    GROUP BY filename
    ORDER BY filename
""").show()

**1.3** — Schema einer Wochendatei mit `DESCRIBE`.

In [ ]:
con.sql("""
    DESCRIBE FROM 'raw/buchungen/woche_01.csv'
""").show()

**Take-away:**
- **Glob-Pattern** ersparen explizite `UNION`-Statements über mehrere Dateien.
- Die **`filename`**-Pseudospalte verrät, woher eine Zeile stammt — unverzichtbar im Lake.
- **`DESCRIBE`** ist die erste Anlaufstelle, um eine unbekannte Datei zu verstehen.

---

## Aufgabe 2 — Schema-Drift entdecken

**Story:** Jetzt willst du **alle** 6 Wochen ansehen — nicht nur die ersten 4. Schau **sehr genau** hin, was DuckDB tut. Vielleicht passiert nicht das, was du erwartest.

**Aufgaben:**
1. Lies alle 6 Wochen mit `woche_*.csv`. Wie viele Zeilen kommen raus, und welche Spalten zeigt `DESCRIBE`?
2. Vergleiche dieses Schema mit dem von `woche_06.csv` einzeln. Was fällt auf?
3. Bring DuckDB dazu, **alle** Spalten aus allen Dateien zu berücksichtigen.

**Hints:**
- Standardmäßig übernimmt DuckDB beim Glob das Schema der **ersten** Datei. Zusätzliche Spalten späterer Dateien werden ohne Fehler verworfen — **stiller Datenverlust!**
- `read_csv()` hat eine Option, die Spalten **über den Namen** matched (statt über die Position) und fehlende mit `NULL` füllt. Schau in die Vorlesungs-Cheatsheet-Folie.

**2.1** — Naive Query über alle 6 Wochen: zähle die Zeilen.

In [ ]:
con.sql("""
    SELECT COUNT(*) AS zeilen
    FROM 'raw/buchungen/woche_*.csv'
""").show()

**2.2** — Welches Schema hat DuckDB beim Glob verwendet?

In [ ]:
con.sql("""
    DESCRIBE FROM 'raw/buchungen/woche_*.csv'
""").show()
# -> Nur 8 Spalten. discount_code fehlt, obwohl die Wochen 5+6 sie haben.

**2.3** — Vergleich: Schema von `woche_06.csv` einzeln.

In [ ]:
con.sql("""
    DESCRIBE FROM 'raw/buchungen/woche_06.csv'
""").show()
# -> 9 Spalten — discount_code ist da. DuckDB hat sie beim Glob still verworfen.

**2.4** — Korrekte Zeilenzahl über alle Wochen, diesmal mit der passenden Option.

In [ ]:
con.sql("""
    SELECT COUNT(*) AS zeilen
    FROM read_csv('raw/buchungen/woche_*.csv', union_by_name=true)
""").show()

**2.5** — Und das jetzt vollständige Schema.

In [ ]:
con.sql("""
    DESCRIBE FROM read_csv('raw/buchungen/woche_*.csv', union_by_name=true)
""").show()

**Take-away:**
- **Schema-Drift** ist ein klassisches Lake-Problem: neue Felder kommen über die Zeit dazu.
- DuckDB warnt **nicht** beim Glob — ohne `DESCRIBE` wäre der Datenverlust unsichtbar.
- **`union_by_name=true`** löst das, indem es Spalten über Namen matcht und Fehlendes mit `NULL` füllt.

---

## Aufgabe 3 — Silver Layer schreiben (`COPY ... TO ... PARQUET`)

**Story:** Die zusammengeführten Buchungen sollen einmal sauber als Parquet im Silver Layer liegen — komprimiert, effizient lesbar.

**Aufgaben:**
1. Schreibe das Ergebnis aus Aufgabe 2 (mit `union_by_name=true`) nach `silver/buchungen.parquet` mit **ZSTD-Kompression**.
2. Prüfe das Schema des frisch geschriebenen Parquet-Files.

**Hints:**
- Geschrieben wird mit `COPY (...) TO '...'`. Im runden Klammer-Block dahinter setzt du `FORMAT` und `COMPRESSION`.
- Den Schema-Check kennst du schon aus Aufgabe 1 — funktioniert genauso auf Parquet.

**3.1** — Buchungen als Parquet mit ZSTD-Kompression nach `silver/` schreiben.

In [ ]:
con.sql("""
    COPY (
        SELECT * FROM read_csv('raw/buchungen/woche_*.csv', union_by_name=true)
    ) TO 'silver/buchungen.parquet' (FORMAT parquet, COMPRESSION zstd)
""")

**3.2** — Schema des frisch geschriebenen Parquet-Files prüfen.

In [ ]:
con.sql("""
    DESCRIBE FROM 'silver/buchungen.parquet'
""").show()

Und der direkte Größenvergleich (vorgegeben):

In [ ]:
import os, glob
csv_size = sum(os.path.getsize(f) for f in glob.glob("raw/buchungen/*.csv"))
par_size = os.path.getsize("silver/buchungen.parquet")
print(f"CSV gesamt: {csv_size / 1024**2:6.2f} MB")
print(f"Parquet:    {par_size / 1024**2:6.2f} MB  (Faktor {csv_size / par_size:.1f}x kleiner)")

**Take-away:**
- **`COPY ... TO ... (FORMAT parquet, COMPRESSION zstd)`** macht aus CSV in einem Schritt ein effizientes Parquet-File.
- Parquet ist **spaltenweise gespeichert + komprimiert** — typisch 5–10× kleiner als CSV.

---

## Aufgabe 4 — Performance: CSV vs. Parquet

**Story:** Lohnt sich der Konvertierungsschritt auch performance-seitig? Wir messen es.

**Aufgaben:** Fülle die **vier SQL-Strings** unten aus. Den Benchmark-Loop und die Auswertung musst du nicht anfassen.
- **Q1:** `SELECT *` über alle Buchungen — einmal von CSV, einmal von Parquet.
- **Q2:** Nur die Spalten `booking_id, price_eur` mit `WHERE price_eur > 20` — wieder beide Varianten.

**Hints:**
- Für CSV brauchst du wieder den Glob mit der `union_by_name`-Option aus Aufgabe 2.
- Bei Parquet darfst du den Pfad direkt hinter `FROM` ansprechen — keine `read_*`-Funktion nötig.

**4.1** — Q1 auf CSV: `SELECT *` über alle Wochen.

In [ ]:
q1_csv = """
    SELECT * FROM read_csv('raw/buchungen/woche_*.csv', union_by_name=true)
"""

**4.2** — Q1 auf Parquet: `SELECT *` vom Silver-File.

In [ ]:
q1_par = """
    SELECT * FROM 'silver/buchungen.parquet'
"""

**4.3** — Q2 auf CSV: nur 2 Spalten, mit `WHERE price_eur > 20`.

In [ ]:
q2_csv = """
    SELECT booking_id, price_eur
    FROM read_csv('raw/buchungen/woche_*.csv', union_by_name=true)
    WHERE price_eur > 20
"""

**4.4** — Q2 auf Parquet: dieselbe Query gegen das Silver-File.

In [ ]:
q2_par = """
    SELECT booking_id, price_eur
    FROM 'silver/buchungen.parquet'
    WHERE price_eur > 20
"""

Und der Benchmark (vorgegeben — nutzt die 4 Queries von oben):

In [ ]:
import time

def benchmark(query, n=3):
    times = []
    for _ in range(n):
        t = time.perf_counter()
        con.sql(query).fetchall()
        times.append(time.perf_counter() - t)
    return min(times)

print(f"{'Query':<34} {'CSV':>10} {'Parquet':>10} {'Faktor':>8}")
print("-" * 64)
for label, qc, qp in [
    ("Q1: SELECT *",                q1_csv, q1_par),
    ("Q2: 2 Spalten + WHERE-Filter", q2_csv, q2_par),
]:
    tc, tp = benchmark(qc), benchmark(qp)
    print(f"{label:<34} {tc:>9.3f}s {tp:>9.3f}s {tc / tp:>7.1f}x")

**Take-away:**
- **Q1 (`SELECT *`):** Vorteil eher klein — alle Daten müssen ohnehin materialisiert werden.
- **Q2 (zwei Spalten + Filter):** Parquet ist drastisch schneller. **Column Pruning** liest nur 2 von 9 Spalten, **Predicate Pushdown** überspringt Row Groups anhand der Statistiken.
- Speicherformat **=** Performance.

---

## Aufgabe 5 — JSON und Parquet joinen

**Story:** Das Marketing will wissen, welche Stationen insgesamt am häufigsten genutzt wurden. Die Stammdaten der Stationen liegen als JSON, die Buchungen als Parquet — DuckDB joint beide direkt ohne Vor-Import.

**Aufgaben:**
1. Schau dir das Schema von `stationen.json` an.
2. Welche **10 Stationen** haben insgesamt die meisten Buchungen? Joine die Buchungen mit den Stationsstammdaten.

**Hints:**
- JSON-Dateien öffnest du analog zu CSV/Parquet — es gibt eine eigene `read_*`-Funktion dafür.
- Die JSON-Funktion verhält sich wie eine Tabelle und kann direkt im `JOIN` stehen.
- Klassisches `JOIN ... ON ...` zwischen Buchungen und Stationen, anschließend `GROUP BY` + `ORDER BY` + `LIMIT`. Keine CTEs nötig.

**5.1** — Schema von `stationen.json`.

In [ ]:
con.sql("""
    DESCRIBE FROM read_json('raw/stationen.json')
""").show()

**5.2** — Top 10 Stationen über alle Buchungen (`JOIN ... ON ...`).

In [ ]:
con.sql("""
    SELECT s.name, s.district, COUNT(*) AS buchungen
    FROM 'silver/buchungen.parquet' b
    JOIN read_json('raw/stationen.json') s
      ON b.station_id = s.station_id
    GROUP BY s.name, s.district
    ORDER BY buchungen DESC
    LIMIT 10
""").show()

**Take-away:**
- DuckDB joint **JSON und Parquet** in derselben Query — kein Vor-Import nötig.
- `read_json('...')` verhält sich wie eine Tabelle und kann direkt im `JOIN` stehen.

---

## Aufgabe 6 (optional) — Partitionierung nach Monat

**Story:** Das Datenvolumen wächst. Damit BI-Queries auf einen einzelnen Monat nicht das ganze Parquet-File lesen müssen, willst du nach Monat partitionieren — pro Monat ein eigener Unterordner.

**Aufgaben:**
1. Schreibe die Buchungen nach `silver/buchungen_partitioned/` und partitioniere nach Monat. Die Spalte `month` musst du dabei aus `start_time` berechnen.
2. Frage **nur die Februar-Partition** ab — über einen Glob-Pfad in den passenden Unterordner.

**Hints:**
- Die `COPY ... TO`-Syntax aus Aufgabe 3 kennst du schon. Im Options-Block gibt's eine zusätzliche Option `PARTITION_BY (...)`.
- Die Partitionsspalte muss im `SELECT` enthalten sein. Den Monat aus einem Timestamp holst du mit `MONTH(...)`.
- DuckDB legt pro Wert einen Ordner an, z. B. `month=1`, `month=2`, ... — den kannst du anschließend per Glob lesen.

**6.1** — Partitioniert nach Monat schreiben.

In [ ]:
con.sql("""
    COPY (
        SELECT *, MONTH(start_time) AS month
        FROM 'silver/buchungen.parquet'
    ) TO 'silver/buchungen_partitioned' (
        FORMAT parquet, PARTITION_BY (month),
        COMPRESSION zstd
    )
""")

Vorgegeben — die erzeugte Verzeichnisstruktur:

In [ ]:
import os
print("Partitionen unter silver/buchungen_partitioned/:")
for entry in sorted(os.listdir("silver/buchungen_partitioned")):
    print(" ", entry)

**6.2** — Nur die Februar-Partition lesen (Glob-Pfad in den passenden Unterordner).

In [ ]:
con.sql("""
    SELECT COUNT(*) AS februar_buchungen
    FROM 'silver/buchungen_partitioned/month=2/*.parquet'
""").show()

**Take-away:**
- **`PARTITION_BY`** legt pro Wert einen Ordner an — Queries mit Filter auf diese Spalte können ganze Partitionen überspringen (**Partition Pruning**).
- Zusammen mit Column Pruning + Predicate Pushdown bekommt man Warehouse-ähnliche Performance auf reinen Files.

---

## Geschafft!

Du hast in dieser Übung:
1. Den Lake mit **Glob-Pattern** und **`DESCRIBE`** erkundet
2. **Schema-Drift** entdeckt und mit **`union_by_name`** gelöst
3. Einen **Silver Layer** als Parquet mit ZSTD-Kompression geschrieben
4. **CSV vs. Parquet** gemessen — Column Pruning + Predicate Pushdown live erlebt
5. **JSON und Parquet** in einer Query gejoint
6. *(Optional)* Eine **partitionierte** Parquet-Tabelle erzeugt

Pipeline `Raw -> Silver -> partitioniertes Silver` ist komplett.